# 00 — Environment setup (conda/mamba) + repository sanity checks

Set up the `rednet-ml` conda environment, register an ipykernel, and verify your repo layout before running the pipeline.


## 0.1 Create / update the `rednet-ml` environment

Run these **in a terminal** (not inside a running notebook kernel):


In [ ]:

# cd /path/to/REDNET-ML
# mamba env create -f cfg/environment.yml -n rednet-ml   # or: conda env create ...
# conda activate rednet-ml
# pip install -r requirements.txt   # if env.yml doesn't include all deps
# python -m ipykernel install --user --name rednet-ml --display-name "rednet-ml"
# jupyter lab


## 0.2 Required environment variables (OBPG)

OBDAAC downloads require an app key.


In [ ]:

import os
print("OBPG_APPKEY set?:", bool(os.environ.get("OBPG_APPKEY")))


## 0.3 Repo layout sanity-check


In [4]:
from __future__ import annotations

import os, sys, subprocess, json, re
from pathlib import Path
from datetime import datetime

REPO_ROOT = Path.cwd().parent
print("REPO_ROOT:", REPO_ROOT)

def sh(cmd: str, check: bool=True) -> None:
    """Run a shell command (prints it first)."""
    print("\n▶", cmd)
    subprocess.run(cmd, shell=True, check=check)

def pick_first_existing(*cands: str) -> Path:
    for c in cands:
        p = Path(c)
        if p.exists():
            return p
    return Path(cands[0])

def require_exists(p: Path, what: str="path") -> Path:
    if not p.exists():
        raise FileNotFoundError(f"Missing {what}: {p}")
    return p

def newest_path(glob_pat: str) -> Path | None:
    paths = list(REPO_ROOT.glob(glob_pat))
    if not paths:
        return None
    paths.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return paths[0]

def show_tree(root: Path, max_lines: int=200) -> None:
    i = 0
    for p in sorted(root.rglob("*")):
        if i >= max_lines:
            print("... (truncated)")
            return
        if p.is_dir():
            continue
        rel = p.relative_to(root)
        print(rel)
        i += 1

expected = [
    "scripts/download",
    "scripts/HAB/preparation",
    "scripts/HAB/compare",
    "scripts/fusion",
    "scripts/eval",
    "deployment/aoi",
    "deployment/src/inference",
    "src/torchvision_det",
    "cfg",
    "runs",
    "data",
]
missing = [p for p in expected if not (REPO_ROOT / p).exists()]
if missing:
    print("⚠️ Missing expected paths:")
    for m in missing: print(" -", m)
else:
    print("✓ Repo layout looks good.")


REPO_ROOT: /Users/ameerfiras/REDNET-ML
✓ Repo layout looks good.


## 0.4 Dependency import smoke test


In [5]:

import numpy as np, pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
print("numpy/pandas/xarray/matplotlib imports OK")

try:
    import torch
    print("torch:", torch.__version__)
except Exception as e:
    print("torch import failed:", e)


numpy/pandas/xarray/matplotlib imports OK
torch: 2.7.1
